In [2]:
from pathlib import Path
import sys

import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

In [3]:
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: /home/kriss/VSProjects/CV_project/CV-P22-building-segmentation


In [4]:
from src.preprocessing import load_and_preprocess


In [5]:
SPLITS_DIR = PROJECT_ROOT / "data" / "splits"

train_df = pd.read_csv(SPLITS_DIR / "train.csv")
val_df = pd.read_csv(SPLITS_DIR / "val.csv")
test_df = pd.read_csv(SPLITS_DIR / "test.csv")


In [6]:
class BuildingDataset(Dataset):
    def __init__(self, dataframe, project_root):
        self.dataframe = dataframe.reset_index(drop=True)
        self.project_root = project_root

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]

        image_path = self.project_root / row["image_path"]
        mask_path = self.project_root / row["mask_path"]

        image, mask = load_and_preprocess(
            image_path,
            mask_path
        )

        return image, mask

In [7]:
train_dataset = BuildingDataset(train_df, PROJECT_ROOT)
val_dataset = BuildingDataset(val_df, PROJECT_ROOT)
test_dataset = BuildingDataset(test_df, PROJECT_ROOT)


In [8]:
BATCH_SIZE = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

In [19]:
%pip install segmentation-models-pytorch

  Using cached safetensors-0.8.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.2 kB)
  Using cached tqdm-4.70.1-py3-none-any.whl.metadata (57 kB)
  Using cached click-8.5.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
  Using cached anyio-4.15.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.0/838.0 kB 6.4 MB/s  0:00:00
Using cached click-8.5.0-py3-none-any.whl (125 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 10.4 MB/s  0:00:00m0:00:0100:01
Using cached anyio-4.15.1-py3-none-any.whl (132 kB)
Using cached h11-0.16.0-py3-none-any.whl (37 kB)
Using cached pyyaml-6.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (807 kB)
Using cached safetensors-0.8.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.

In [9]:
import segmentation_models_pytorch as smp

/home/kriss/VSProjects/sticker-sales-forecasting/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


Device: cpu


In [11]:
model = smp.Unet(
    encoder_name="resnet18",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1
)

model = model.to(device)

print("Baseline model created.")

Baseline model created.


In [12]:
images, masks = next(iter(train_loader))

images = images.to(device)
masks = masks.to(device)

with torch.no_grad():
    outputs = model(images)


In [13]:
criterion = torch.nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

In [14]:
def compute_segmentation_stats(logits, targets, threshold=0.5):
    probabilities = torch.sigmoid(logits)
    predictions = (probabilities >= threshold).float()

    intersection = (predictions * targets).sum().item()

    prediction_sum = predictions.sum().item()
    target_sum = targets.sum().item()

    union = prediction_sum + target_sum - intersection

    return intersection, union, prediction_sum, target_sum

In [15]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0

    total_intersection = 0.0
    total_union = 0.0
    total_prediction_sum = 0.0
    total_target_sum = 0.0

    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, masks)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

        intersection, union, pred_sum, target_sum = (
            compute_segmentation_stats(outputs.detach(), masks)
        )

        total_intersection += intersection
        total_union += union
        total_prediction_sum += pred_sum
        total_target_sum += target_sum

    average_loss = total_loss / len(loader.dataset)

    iou = total_intersection / (total_union + 1e-7)

    dice = (
        2 * total_intersection
        / (total_prediction_sum + total_target_sum + 1e-7)
    )

    return average_loss, iou, dice

In [16]:
def evaluate(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0

    total_intersection = 0.0
    total_union = 0.0
    total_prediction_sum = 0.0
    total_target_sum = 0.0

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)

            loss = criterion(outputs, masks)

            total_loss += loss.item() * images.size(0)

            intersection, union, pred_sum, target_sum = (
                compute_segmentation_stats(outputs, masks)
            )

            total_intersection += intersection
            total_union += union
            total_prediction_sum += pred_sum
            total_target_sum += target_sum

    average_loss = total_loss / len(loader.dataset)

    iou = total_intersection / (total_union + 1e-7)

    dice = (
        2 * total_intersection
        / (total_prediction_sum + total_target_sum + 1e-7)
    )

    return average_loss, iou, dice

In [17]:
NUM_EPOCHS = 10

history = []

best_val_loss = float("inf")
best_model_path = PROJECT_ROOT / "results" / "baseline_unet_resnet18.pt"

best_model_path.parent.mkdir(parents=True, exist_ok=True)

In [18]:
for epoch in range(1, NUM_EPOCHS + 1):

    train_loss, train_iou, train_dice = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    val_loss, val_iou, val_dice = evaluate(
        model,
        val_loader,
        criterion,
        device
    )

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_iou": train_iou,
        "train_dice": train_dice,
        "val_loss": val_loss,
        "val_iou": val_iou,
        "val_dice": val_dice,
    })

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
        f"train loss: {train_loss:.4f} | "
        f"val loss: {val_loss:.4f} | "
        f"val IoU: {val_iou:.4f} | "
        f"val Dice: {val_dice:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_model_path)

KeyboardInterrupt: 